# Prior-knowledge-source ablation

In [ ]:
mode = "subset"


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

colors = [
    "#70acc0",
    "#c12075",
    "#003754",
    "#4c626c",
    "#be9e7c",
    "#723c5d",
    "#9d9c9c",
    "#00695c",
    "#e57373",
    "#000000",
    "#ffcc00",
    "#8b4513",
]
colors_name = [
    "graublau",
    "purpur",
    "blau",
    "dunkelgrau",
    "bronze",
    "violett",
    "hellgrau",
    "gedämpftes teal",
    "sanftes koralle",
    "schwarz",
    "goldgelb",
    "sattelbraun",
]
sns.set_palette(sns.color_palette(colors))
base_color = "#70acc0"
light_palette = sns.light_palette(base_color, as_cmap=False)
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)
colors_heat = ["#c12075", "#f0f0f0", "#003754"]
cmap_heat = LinearSegmentedColormap.from_list("custom_diverging", colors_heat, N=256)
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

import re as _re
import matplotlib.figure as _mpl_figure

def _fix_svg_font_declarations(_svg_path):
    """Convert matplotlib's unsupported CSS 'font:' shorthand (labeled px, but
    actually point-sized, in a pt-calibrated viewBox) into explicit
    font-family/font-size/font-weight/font-style, which is what Figma's SVG
    importer requires to resolve size and style correctly."""
    try:
        with open(_svg_path, "r", encoding="utf-8") as _f:
            _content = _f.read()
    except OSError:
        return
    def _convert(_m):
        _oblique = _m.group(1)
        _weight = _m.group(2)
        _pt = float(_m.group(3)) * 0.75
        _family = _m.group(4)
        _font_weight = "bold" if _weight else "normal"
        _font_style = "oblique" if _oblique else "normal"
        return (
            f"font-family: {_family}; font-size: {_pt:g}pt; "
            f"font-weight: {_font_weight}; font-style: {_font_style}"
        )
    _pattern = _re.compile(r'font:\s*(oblique\s+)?(?:(\d+)\s+)?([0-9.]+)px\s+([^;"]+)')
    _new_content = _pattern.sub(_convert, _content)
    if _new_content != _content:
        with open(_svg_path, "w", encoding="utf-8") as _f:
            _f.write(_new_content)

_orig_figure_savefig = _mpl_figure.Figure.savefig

def _rasterize_dense_elements(_fig):
    for _ax in _fig.get_axes():
        for _coll in list(_ax.collections):
            try:
                _coll.set_rasterized(True)
            except Exception:
                pass
        for _im in _ax.get_images():
            try:
                _im.set_rasterized(True)
            except Exception:
                pass

def _patched_figure_savefig(self, fname, *args, **kwargs):
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _rasterize_dense_elements(self)
        kwargs.setdefault("dpi", 300)
    _result = _orig_figure_savefig(self, fname, *args, **kwargs)
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _fix_svg_font_declarations(fname)
    return _result

_mpl_figure.Figure.savefig = _patched_figure_savefig
mpl.rcParams["figure.dpi"] = 500.0
mpl.rcParams["savefig.transparent"] = True
mpl.rcParams["savefig.format"] = "png"

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]

mpl.rcParams["axes.edgecolor"] = "black"
mpl.rcParams["axes.linewidth"] = 0.8
mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.color"] = "grey"
mpl.rcParams["grid.linestyle"] = "--"
mpl.rcParams["grid.linewidth"] = 0.5

mpl.rcParams["legend.frameon"] = False
mpl.rcParams["legend.loc"] = "best"

mpl.rcParams["xtick.major.size"] = 3
mpl.rcParams["xtick.minor.size"] = 2
mpl.rcParams["ytick.major.size"] = 3
mpl.rcParams["ytick.minor.size"] = 2
mpl.rcParams["xtick.major.width"] = 0.4
mpl.rcParams["xtick.minor.width"] = 0.3
mpl.rcParams["ytick.major.width"] = 0.4
mpl.rcParams["ytick.minor.width"] = 0.3
scaling_factor = 1
if mode == "all":
    base_font_size = 4
else:
    base_font_size = 6
base_axes_title_size = base_font_size + 1
base_axes_label_size = base_font_size
base_tick_label_size = base_font_size - 0.5
base_legend_font_size = base_font_size - 0.5
base_figure_title_size = base_font_size + 2
mpl.rcParams["font.size"] = base_font_size * scaling_factor
mpl.rcParams["axes.titlesize"] = base_axes_title_size * scaling_factor
mpl.rcParams["axes.labelsize"] = base_axes_label_size * scaling_factor
mpl.rcParams["xtick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["ytick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["legend.fontsize"] = base_legend_font_size * scaling_factor
mpl.rcParams["figure.titlesize"] = base_figure_title_size * scaling_factor

mpl.rcParams["patch.linewidth"] = 0.7

mpl.rcParams["figure.autolayout"] = True


In [ ]:
def set_figure_params(column, height):

    single_column_width = 3.5
    double_column_width = 7.0

    if column == "single":
        figure_width = single_column_width
    elif column == "double":
        figure_width = double_column_width
    else:
        figure_width = column
    scaling_factor = figure_width / single_column_width
    scaling_factor = 1

    mpl.rcParams["figure.figsize"] = (figure_width, height)


In [ ]:
import sys
import os

_OUTDIR_ABS = os.path.join(os.getcwd(), "outputs", "fig2b_neurips2021_cite_bmmc_scib_ablations")
os.makedirs(_OUTDIR_ABS, exist_ok=True)

sys.path.append(os.path.abspath("../.experiment_data/primary/benchmarking"))

import contextlib
import functools

_POSTPROC_CWD = os.getcwd()

@contextlib.contextmanager
def _in_primary_dir():
    os.chdir(os.path.join(_POSTPROC_CWD, "../.experiment_data/primary"))
    try:
        yield
    finally:
        os.chdir(_POSTPROC_CWD)

with _in_primary_dir():
    import muon as _muon
    _orig_sample_obs = _muon.pp.sample_obs
    def _patched_sample_obs(data, *a, **kw):
        return _orig_sample_obs(data, *a, **kw).copy()
    _muon.pp.sample_obs = _patched_sample_obs
    import datasets

for _name in dir(datasets):
    if _name.startswith("load_"):
        def _make_wrapper(fn):
            @functools.wraps(fn)
            def _wrapper(*a, **kw):
                with _in_primary_dir():
                    return fn(*a, **kw)
            return _wrapper
        setattr(datasets, _name, _make_wrapper(getattr(datasets, _name)))
import glob
import pandas as pd
import scipy
import numpy as np
import os
from tqdm import tqdm
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from scipy.spatial.distance import euclidean
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pickle
import glob
from collections import Counter
from pathlib import Path
from goatools.obo_parser import GODag
from goatools.anno.gaf_reader import GafReader
import glob
import pandas as pd
import scipy
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
import textwrap
from adjustText import adjust_text
import matplotlib.patches as mpatches
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


In [ ]:
def show_umap_plot(X_umap, labels, cmap, encode_first_as_grey=False, continuous=False, title='', figsize=(3, 3), save_name=False):
    set_figure_params(4, 4)
    if not continuous:
        unique_labels, encoded_labels = np.unique(labels, return_inverse=True)
        n_categories = len(unique_labels)
        base_palette = sns.color_palette('husl', n_categories)
        if n_categories > 10:
            reordered_palette = []
            spacing = 6
            for i in range(len(base_palette)):
                index = i // spacing + i % spacing * (len(base_palette) // spacing)
                reordered_palette.append(base_palette[index % n_categories])
        else:
            reordered_palette = base_palette
        if not encode_first_as_grey:
            new_palette = reordered_palette
            cmap = ListedColormap(new_palette)
        else:
            new_palette = ['#eff1f2'] + reordered_palette[1:]
            cmap = ListedColormap(new_palette)
    else:
        encoded_labels = labels
    fig, ax = plt.subplots()
    ax.set_aspect('equal', 'box')
    if continuous:
        sc = ax.scatter(X_umap[:, 0], X_umap[:, 1], s=0.1, cmap=cmap, c=encoded_labels, edgecolor='none')
    else:
        sc = ax.scatter(X_umap[:, 0], X_umap[:, 1], s=0.1, c=encoded_labels, cmap=cmap, edgecolor='none')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_box_aspect(1)
    plt.title(title)
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()
    plt.close()

def show_umap_plot(X_umap, labels, cmap, encode_first_as_grey=False, continuous=False, title='', figsize=(3, 3), save_name=False):
    set_figure_params(4, 4)
    unique_labels = np.unique(labels)
    label_to_int = {label: i for i, label in enumerate(unique_labels)}
    encoded_labels = [label_to_int[l] for l in labels]
    colors = [type_to_color.get(label, '#9E9E9E') for label in unique_labels]
    cmap = ListedColormap(colors)
    fig, ax = plt.subplots()
    ax.set_aspect('equal', 'box')
    sc = ax.scatter(X_umap[:, 0], X_umap[:, 1], s=0.1, c=encoded_labels, cmap=cmap, edgecolor='none')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_box_aspect(1)
    plt.title(title)
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()
    plt.close()

def show_colorbar(cmap, save_name):
    fig, ax = plt.subplots(figsize=(1.5, 4))
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=1))
    cbar = plt.colorbar(sm, ax=ax)
    cbar.ax.tick_params(labelsize=10)
    cbar.set_label('GO activation', rotation=270, labelpad=15)
    plt.axis('off')
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()

def show_legend(labels, save_name):
    unique_labels, encoded_labels = np.unique(labels, return_inverse=True)
    n_categories = len(unique_labels)
    base_palette = sns.color_palette('husl', n_categories)
    if n_categories > 10:
        reordered_palette = []
        spacing = 6
        for i in range(len(base_palette)):
            index = i // spacing + i % spacing * (len(base_palette) // spacing)
            reordered_palette.append(base_palette[index % n_categories])
    else:
        reordered_palette = base_palette
    new_palette = reordered_palette
    cmap = ListedColormap(new_palette)
    fig, ax = plt.subplots(figsize=(2, 8))
    handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=cmap(i)) for i in range(n_categories)]
    ax.legend(handles, unique_labels, bbox_to_anchor=(1, 1), loc='upper left')
    plt.axis('off')
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()

def show_legend(labels, save_name=False):
    unique_labels = sorted(set(labels))
    colors = [type_to_color.get(label, '#9E9E9E') for label in unique_labels]
    fig, ax = plt.subplots(figsize=(2, min(0.3 * len(unique_labels), 15)))
    handles = [plt.Line2D([0], [0], marker='o', linestyle='None', markersize=6, markerfacecolor=color, markeredgecolor='k') for color in colors]
    ax.legend(handles, unique_labels, bbox_to_anchor=(1, 1), loc='upper left', frameon=False)
    plt.axis('off')
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()
    plt.close()

def show_legend(labels, save_name=False):
    family_to_labels = defaultdict(list)
    for label in np.unique(labels):
        family = map_cell_type_to_family(label)
        family_to_labels[family].append(label)
    family_order = list(family_colors.keys())
    handles = []
    legend_labels = []
    for family in family_order:
        for label in sorted(family_to_labels.get(family, [])):
            color = type_to_color.get(label, '#9E9E9E')
            handles.append(plt.Line2D([0], [0], marker='o', linestyle='None', markersize=6, markerfacecolor=color, markeredgecolor='k'))
            legend_labels.append(f'{label}')
    fig, ax = plt.subplots()
    ax.legend(handles, legend_labels, bbox_to_anchor=(1, 1), loc='upper left', frameon=False)
    plt.axis('off')
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()
    plt.close()


In [ ]:
def generate_results_tables(results, save_name=None):
    for hyper_label, hyper_bool in zip(['default', 'hyper'], [False, True]):
        results_filtered = results[(results['scib_type'] == 'combined') & (results['hyper'] == False)]
        results_table = results_filtered.pivot(index='tool_label', columns='paired_rate', values='scib')
        results_table = results_table[sorted(results_table.columns, reverse=True)]
        unique_elements, indices = np.unique(results_filtered['tool_label'], return_index=True)
        custom_index_order = unique_elements[np.argsort(indices)]
        results_table = results_table.reindex(custom_index_order)
        results_table.to_csv(save_name + f'_{hyper_label}_results.table.csv')

def generate_results_plot(results, paired_rates, tools_labels):
    results_plot = []
    for paired_rate in paired_rates:
        print(paired_rate)
        results_plot_sub = []
        for tool_label in tools_labels:
            if tool_label == 'WNN' or tool_label == 'MIDAS' or tool_label == 'MOFA+':
                hyper_bool = False
            else:
                hyper_bool = True
            print(results[(results['paired_rate'] == paired_rate) & (results['tool_label'] == tool_label)])
            print(paired_rate)
            print(tool_label)
            result_plot_hyper_label = results[(results['paired_rate'] == paired_rate) & (results['tool_label'] == tool_label) & (results['scib_type'] == 'label_adj') & (results['hyper'] == hyper_bool)]['scib']
            result_plot_hyper_label = result_plot_hyper_label.iloc[0] if len(result_plot_hyper_label) > 0 else 0
            result_plot_hyper_modality = results[(results['paired_rate'] == paired_rate) & (results['tool_label'] == tool_label) & (results['scib_type'] == 'modality_adj') & (results['hyper'] == hyper_bool)]['scib']
            result_plot_hyper_modality = result_plot_hyper_modality.iloc[0] if len(result_plot_hyper_modality) > 0 else 0
            result_plot_hyper_batch = results[(results['paired_rate'] == paired_rate) & (results['tool_label'] == tool_label) & (results['scib_type'] == 'batch_adj') & (results['hyper'] == hyper_bool)]['scib']
            result_plot_hyper_batch = result_plot_hyper_batch.iloc[0] if len(result_plot_hyper_batch) > 0 else 0
            print(np.array([result_plot_hyper_label, result_plot_hyper_modality, result_plot_hyper_batch]))
            results_plot_sub.append(np.array([result_plot_hyper_label, result_plot_hyper_modality, result_plot_hyper_batch]))
        results_plot.append(np.array(results_plot_sub))
    return results_plot

def generate_results_plot_old(results, paired_rates, tools_labels):
    results_plot = []
    for paired_rate in paired_rates:
        print(paired_rate)
        results_plot_sub = []
        for tool_label in tools_labels:
            print(results[(results['paired_rate'] == paired_rate) & (results['tool_label'] == tool_label)])
            print(paired_rate)
            print(tool_label)
            result_plot_label = results[(results['paired_rate'] == paired_rate) & (results['tool_label'] == tool_label) & (results['scib_type'] == 'label_adj') & (results['hyper'] == False)]['scib']
            result_plot_label = result_plot_label.iloc[0] if len(result_plot_label) > 0 else 0
            result_plot_hyper_label = results[(results['paired_rate'] == paired_rate) & (results['tool_label'] == tool_label) & (results['scib_type'] == 'label_adj') & (results['hyper'] == True)]['scib']
            result_plot_hyper_label = result_plot_hyper_label.iloc[0] if len(result_plot_hyper_label) > 0 else 0
            result_plot_hyper_label = max(0, result_plot_hyper_label - result_plot_label)
            result_plot_modality = results[(results['paired_rate'] == paired_rate) & (results['tool_label'] == tool_label) & (results['scib_type'] == 'modality_adj') & (results['hyper'] == False)]['scib']
            result_plot_modality = result_plot_modality.iloc[0] if len(result_plot_modality) > 0 else 0
            result_plot_hyper_modality = results[(results['paired_rate'] == paired_rate) & (results['tool_label'] == tool_label) & (results['scib_type'] == 'modality_adj') & (results['hyper'] == True)]['scib']
            result_plot_hyper_modality = result_plot_hyper_modality.iloc[0] if len(result_plot_hyper_modality) > 0 else 0
            result_plot_hyper_modality = max(0, result_plot_hyper_modality - result_plot_modality)
            result_plot_batch = results[(results['paired_rate'] == paired_rate) & (results['tool_label'] == tool_label) & (results['scib_type'] == 'batch_adj') & (results['hyper'] == False)]['scib']
            result_plot_batch = result_plot_batch.iloc[0] if len(result_plot_batch) > 0 else 0
            result_plot_hyper_batch = results[(results['paired_rate'] == paired_rate) & (results['tool_label'] == tool_label) & (results['scib_type'] == 'batch_adj') & (results['hyper'] == True)]['scib']
            result_plot_hyper_batch = result_plot_hyper_batch.iloc[0] if len(result_plot_hyper_batch) > 0 else 0
            result_plot_hyper_batch = max(0, result_plot_hyper_batch - result_plot_batch)
            print(np.array([result_plot_label, result_plot_modality, result_plot_batch, result_plot_hyper_label, result_plot_hyper_modality, result_plot_hyper_batch]))
            results_plot_sub.append(np.array([result_plot_label, result_plot_modality, result_plot_batch, result_plot_hyper_label, result_plot_hyper_modality, result_plot_hyper_batch]))
        results_plot.append(np.array(results_plot_sub))
    return results_plot

def plot_performance_old(tools_labels, results, results_plot, bar_width=0.05, bar_spacing=0, figsize=(14, 7), show_legend=True, save_name=False):
    set_figure_params('single', 2.5)

    def generate_patterns(n_parts):
        if n_parts == 4:
            return ['//', None, '//', None]
        elif n_parts == 6:
            return ['//', 'oo', '..', '//', 'oo', '..']

    def lighten_color(color, amount=0.6):
        c = np.array(mcolors.to_rgba(color))
        white = np.array([1, 1, 1, 1])
        return tuple((1 - amount) * c + amount * white)
    groups = [len(tools_labels)] + [len(tools_labels) - 2] * 4
    n_parts = 6
    patterns = generate_patterns(n_parts)
    data = results_plot
    base_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf', '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#c5b0d5', '#c49c94', '#f7b6d2', '#c7c7c7', '#dbdb8d', '#9edae5']
    base_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    light_colors = [lighten_color(color, amount=0.4) for color in base_colors]
    fig, ax = plt.subplots()
    bar_width = bar_width
    indices = np.arange(len(paired_rates))
    group_centers = []
    for i, group_data in enumerate(data):
        group_center = indices[i] + (bar_width + bar_spacing) * (groups[i] - 1) / 2
        group_centers.append(group_center)
        for j in range(groups[i]):
            bottom = np.zeros(group_data.shape[1])
            for k in range(group_data.shape[1]):
                color = base_colors[j] if k < n_parts // 2 else light_colors[j]
                bar_position = indices[i] + j * (bar_width + bar_spacing)
                ax.bar(bar_position, group_data[j, k], bar_width, bottom=bottom, color=color, capsize=3, edgecolor='black', hatch=patterns[k])
                bottom += group_data[j, k]
    ax.set_xticks(group_centers)
    ax.set_xticklabels([f'{pr:.1f}' for pr in paired_rates])
    ax.set_xlabel('paired rate')
    ax.set_ylabel('combined scib')
    ax.set_ylim(bottom=0, top=max(results[results['scib_type'] == 'combined']['scib']) + 0.05)
    legend_handles_tools = [plt.Rectangle((0, 0), 1, 1, color=color, edgecolor='black') for color in base_colors]
    if show_legend:
        ax.legend(legend_handles_tools, tools_labels, loc='upper left', bbox_to_anchor=(1, 1))
    plt.tight_layout()
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()

def plot_performance(tools_labels, results, results_plot, bar_width=0.05, bar_spacing=0, figsize=(14, 7), show_legend=True, save_name=False):
    set_figure_params('single', 2)

    def lighten_color(color, amount=0.6):
        c = np.array(mcolors.to_rgba(color))
        white = np.array([1, 1, 1, 1])
        return tuple((1 - amount) * c + amount * white)
    groups = [len(tools_labels)] + [len(tools_labels) - 4] * 4
    data = results_plot
    base_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf', '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#c5b0d5', '#c49c94', '#f7b6d2', '#c7c7c7', '#dbdb8d', '#9edae5']
    biological_preseveration_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    batch_colors = [lighten_color(color, amount=0.6) for color in biological_preseveration_colors]
    modality_colors = [lighten_color(color, amount=0.3) for color in biological_preseveration_colors]
    colors = [biological_preseveration_colors, modality_colors, batch_colors]
    fig, ax = plt.subplots()
    bar_width = bar_width
    indices = np.arange(len(paired_rates))
    group_centers = []
    for i, group_data in enumerate(data):
        group_center = indices[i] + (bar_width + bar_spacing) * (groups[i] - 1) / 2
        group_centers.append(group_center)
        for j in range(groups[i]):
            bottom = np.zeros(group_data.shape[1])
            for k in range(group_data.shape[1]):
                color = colors[k][j]
                bar_position = indices[i] + j * (bar_width + bar_spacing)
                ax.bar(bar_position, group_data[j, k], bar_width, bottom=bottom, color=color, capsize=3, edgecolor='black')
                bottom += group_data[j, k]
    ax.set_xticks(group_centers)
    ax.set_xticklabels([f'{pr:.1f}' for pr in paired_rates])
    ax.set_xlabel('paired rate')
    ax.set_ylabel('combined scib')
    ax.set_ylim(bottom=0, top=max(results[results['scib_type'] == 'combined']['scib']) + 0.05)
    legend_handles_tools = [plt.Rectangle((0, 0), 1, 1, color=color, edgecolor='black') for color in biological_preseveration_colors[:len(tools_labels)]]
    scib_legend_handles_tools = [plt.Rectangle((0, 0), 1, 1, color='white', edgecolor='white')] + [plt.Rectangle((0, 0), 1, 1, color=lighten_color(biological_preseveration_colors[0], amount=lighten_amount), edgecolor='black') for lighten_amount in [0, 0.3, 0.6]]
    scib_labels = ['', 'biological preservation', 'modality alignment', 'batch correction']
    if show_legend:
        ax.legend(legend_handles_tools + scib_legend_handles_tools, tools_labels + scib_labels, loc='upper left', bbox_to_anchor=(1, 1))
    plt.tight_layout()
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()


In [ ]:
def get_best_hyper(path):

    num_hyper = sum(
        os.path.isdir(os.path.join(path, "hyper", entry))
        for entry in os.listdir(os.path.join(path, "hyper"))
    )
    scib_hyper_labels = list(
        glob.glob(os.path.join(path, "hyper", "hyper*", "scib_results_label.csv"))
    )
    scib_hyper_label = np.array(
        [
            float(pd.read_csv(path_label)["Bio conservation"].iloc[0])
            for path_label in scib_hyper_labels
        ]
    )

    if len(scib_hyper_label) > 0:
        if paired_rate != 1 and paired_rate != 1.0:
            scib_hyper_modality = np.array(
                [
                    float(pd.read_csv(path_label)["Batch correction"].iloc[0])
                    for path_label in glob.glob(
                        os.path.join(path, "hyper", "hyper*", "scib_results_modality.csv")
                    )
                ]
            )
        else:
            scib_hyper_modality = [None] * len(scib_hyper_labels)
        if batch_key:
            scib_hyper_batch = np.array(
                [
                    float(pd.read_csv(path_label)["Batch correction"].iloc[0])
                    for path_label in glob.glob(
                        os.path.join(path, "hyper", "hyper*", f"scib_results_{batch_key}.csv")
                    )
                ]
            )
        else:
            scib_hyper_batch = [None] * len(scib_hyper_labels)

        if batch_key:
            if paired_rate != 1 and paired_rate != 1.0:
                scib_hyper = (
                    0.6 * scib_hyper_label + 0.2 * scib_hyper_modality + 0.2 * scib_hyper_batch
                )
            else:
                scib_hyper = 0.6 * scib_hyper_label + 0.4 * scib_hyper_batch
        else:
            if paired_rate != 1 and paired_rate != 1.0:
                scib_hyper = 0.6 * scib_hyper_label + 0.4 * scib_hyper_modality
            else:
                scib_hyper = scib_hyper_label
        pos_scib_hyper = np.where(scib_hyper == scib_hyper.max())[0][0]
        path_scib_hyper = str(
            Path(
                glob.glob(os.path.join(path, "hyper", "hyper*", "scib_results_label.csv"))[
                    pos_scib_hyper
                ]
            ).parent
        )
    else:
        print("Issue with path")
        num_hyper = None
        path_scib_hyper = None
        scib_hyper = None

    return num_hyper, path_scib_hyper, scib_hyper.max()


In [ ]:
def get_scib_results(path_default_hyper, results, batch_key, tool_label="", hyper=False):
    _visualize = False
    X_umap = np.load(os.path.join(path_default_hyper, "X_umap.npy"))
    try:
        labels = np.load(os.path.join(path_default_hyper, "labels.npy"), allow_pickle=True)[
            train_indices
        ]
    except IndexError:
        labels = np.load(os.path.join(path_default_hyper, "labels.npy"), allow_pickle=True)
    show_umap_plot(
        X_umap,
        labels,
        cmap,
        continuous=False,
        save_name=(f"{_OUTDIR_ABS}/{dataset_name}_labels_{tool_label}_{paired_rate}" if _visualize else False),
    )
    scib_label = float(
        pd.read_csv(os.path.join(path_default_hyper, "scib_results_label.csv"))[
            "Bio conservation"
        ].iloc[0]
    )
    results = pd.concat(
        [
            results,
            pd.DataFrame(
                [[scib_label, "label", tool_label, paired_rate, hyper]],
                columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
            ),
        ],
        ignore_index=True,
    )

    if paired_rate != 1 and paired_rate != 1.0:
        try:
            modality = np.load(os.path.join(path_default_hyper, "modality.npy"), allow_pickle=True)[
                train_indices
            ]
        except IndexError:
            modality = np.load(os.path.join(path_default_hyper, "modality.npy"), allow_pickle=True)

        if "sparsevi" in tool_label.lower():
            X_umap_expression = np.load(os.path.join(path_default_hyper, "X_umap_expression.npy"))
            show_umap_plot(
                X_umap_expression,
                labels,
                cmap,
                continuous=False,
                save_name=(f"{_OUTDIR_ABS}/{dataset_name}_labels_{tool_label}_{paired_rate}_expression" if _visualize else False),
            )
            show_umap_plot(
                X_umap_expression,
                modality,
                cmap,
                continuous=False,
                save_name=(f"{_OUTDIR_ABS}/{dataset_name}_modality_{tool_label}_{paired_rate}_expression" if _visualize else False),
            )
            X_umap_protein = np.load(os.path.join(path_default_hyper, "X_umap_protein.npy"))
            show_umap_plot(
                X_umap_protein,
                labels,
                cmap,
                continuous=False,
                save_name=(f"{_OUTDIR_ABS}/{dataset_name}_labels_{tool_label}_{paired_rate}_protein" if _visualize else False),
            )
            show_umap_plot(
                X_umap_protein,
                modality,
                cmap,
                continuous=False,
                save_name=(f"{_OUTDIR_ABS}/{dataset_name}_modality_{tool_label}_{paired_rate}_protein" if _visualize else False),
            )

        show_umap_plot(
            X_umap,
            modality,
            cmap,
            continuous=False,
            save_name=(f"{_OUTDIR_ABS}/{dataset_name}_modality_{tool_label}_{paired_rate}" if _visualize else False),
        )
        scib_modality = float(
            pd.read_csv(os.path.join(path_default_hyper, "scib_results_modality.csv"))[
                "Batch correction"
            ].iloc[0]
        )
        results = pd.concat(
            [
                results,
                pd.DataFrame(
                    [[scib_modality, "modality", tool_label, paired_rate, hyper]],
                    columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
                ),
            ],
            ignore_index=True,
        )

    if batch_key:
        try:
            batch = np.load(os.path.join(path_default_hyper, f"batch.npy"), allow_pickle=True)[
                train_indices
            ]
        except IndexError:
            batch = np.load(os.path.join(path_default_hyper, f"batch.npy"), allow_pickle=True)

        if "sparsevi" in tool_label.lower():
            if paired_rate != 1 and paired_rate != 1.0:
                show_umap_plot(
                    X_umap_expression,
                    batch,
                    cmap,
                    continuous=False,
                    save_name=(f"{_OUTDIR_ABS}/{dataset_name}_batch_{tool_label}_{paired_rate}_expression" if _visualize else False),
                )
                show_umap_plot(
                    X_umap_protein,
                    batch,
                    cmap,
                    continuous=False,
                    save_name=(f"{_OUTDIR_ABS}/{dataset_name}_batch_{tool_label}_{paired_rate}_protein" if _visualize else False),
                )

        show_umap_plot(
            X_umap,
            batch,
            cmap,
            continuous=False,
            save_name=(f"{_OUTDIR_ABS}/{dataset_name}_{batch_key}_{tool_label}_{paired_rate}" if _visualize else False),
        )
        scib_batch = float(
            pd.read_csv(os.path.join(path_default_hyper, f"scib_results_{batch_key}.csv"))[
                "Batch correction"
            ].iloc[0]
        )
        results = pd.concat(
            [
                results,
                pd.DataFrame(
                    [[scib_batch, "batch", tool_label, paired_rate, hyper]],
                    columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
                ),
            ],
            ignore_index=True,
        )
        if paired_rate != 1 and paired_rate != 1.0:
            scib = 0.6 * scib_label + 0.2 * scib_modality + 0.2 * scib_batch
            results = pd.concat(
                [
                    results,
                    pd.DataFrame(
                        [[scib_label * 0.6, "label_adj", tool_label, paired_rate, hyper]],
                        columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
                    ),
                ],
                ignore_index=True,
            )
            results = pd.concat(
                [
                    results,
                    pd.DataFrame(
                        [[scib_modality * 0.2, "modality_adj", tool_label, paired_rate, hyper]],
                        columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
                    ),
                ],
                ignore_index=True,
            )
            results = pd.concat(
                [
                    results,
                    pd.DataFrame(
                        [[scib_batch * 0.2, "batch_adj", tool_label, paired_rate, hyper]],
                        columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
                    ),
                ],
                ignore_index=True,
            )

        else:
            scib = 0.6 * scib_label + 0.4 * scib_batch
            results = pd.concat(
                [
                    results,
                    pd.DataFrame(
                        [[scib_label * 0.6, "label_adj", tool_label, paired_rate, hyper]],
                        columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
                    ),
                ],
                ignore_index=True,
            )
            results = pd.concat(
                [
                    results,
                    pd.DataFrame(
                        [[scib_batch * 0.4, "batch_adj", tool_label, paired_rate, hyper]],
                        columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
                    ),
                ],
                ignore_index=True,
            )
    else:
        if paired_rate != 1 and paired_rate != 1.0:
            scib = 0.6 * scib_label + 0.4 * scib_modality
            results = pd.concat(
                [
                    results,
                    pd.DataFrame(
                        [[scib_label * 0.6, "label_adj", tool_label, paired_rate, hyper]],
                        columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
                    ),
                ],
                ignore_index=True,
            )
            results = pd.concat(
                [
                    results,
                    pd.DataFrame(
                        [[scib_modality * 0.4, "modality_adj", tool_label, paired_rate, hyper]],
                        columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
                    ),
                ],
                ignore_index=True,
            )
        else:
            scib = scib_label
            results = pd.concat(
                [
                    results,
                    pd.DataFrame(
                        [[scib_label, "label_adj", tool_label, paired_rate, hyper]],
                        columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
                    ),
                ],
                ignore_index=True,
            )
    results = pd.concat(
        [
            results,
            pd.DataFrame(
                [[scib, "combined", tool_label, paired_rate, hyper]],
                columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
            ),
        ],
        ignore_index=True,
    )

    return results


In [ ]:
import sys
import os

sys.path.append(os.path.abspath("../.experiment_data/primary/benchmarking"))
from metrics import calculate_scib_metrics
import anndata as ad

n_bootstrap = 2

def calculate_scib_bootstrapped(
    path_scib_hyper, results, tool_label="", paired_rate=None, hyper=True
):

    if hyper:
        X_latent_addix = path_scib_hyper.split("/")[-3].split("_")[0]
    else:
        X_latent_addix = path_scib_hyper.split("/")[-2].split("_")[0]
    X_latent = np.load(os.path.join(path_scib_hyper, f"X_{X_latent_addix}.npy"))
    try:
        labels = np.load(os.path.join(path_scib_hyper, "labels.npy"), allow_pickle=True)[
            train_indices
        ]
    except IndexError:
        labels = np.load(os.path.join(path_scib_hyper, "labels.npy"), allow_pickle=True)

    if paired_rate != 1 and paired_rate != 1.0:
        try:
            modality = np.load(os.path.join(path_scib_hyper, "modality.npy"), allow_pickle=True)[
                train_indices
            ]
        except IndexError:
            modality = np.load(os.path.join(path_scib_hyper, "modality.npy"), allow_pickle=True)
    else:
        modality = np.zeros_like(labels)

    if batch_key:
        try:
            batch = np.load(os.path.join(path_scib_hyper, f"batch.npy"), allow_pickle=True)[
                train_indices
            ]
        except IndexError:
            batch = np.load(os.path.join(path_scib_hyper, f"batch.npy"), allow_pickle=True)
    else:
        batch = np.zeros_like(labels)

    scib_bootstrapped = []
    for _ in range(n_bootstrap):
        indices = np.random.choice(len(labels), size=int(0.95 * len(labels)), replace=False)
        X_latent_bootstrapped = X_latent[indices]
        fake_anndata = ad.AnnData(
            X=X_latent_bootstrapped,
            obs=pd.DataFrame(
                data={
                    labels_key: labels[indices],
                    "modality": modality[indices],
                    batch_key: batch[indices],
                }
            ),
        )
        scib_bootstrapped.append(
            calculate_scib_metrics(
                None,
                X_latent_bootstrapped,
                fake_anndata,
                labels_key,
                modality_key="modality" if paired_rate != 1 and paired_rate != 1.0 else None,
                batch_key=batch_key,
            )
        )

    np.percentile(scib_bootstrapped, 2.5)
    np.percentile(scib_bootstrapped, 97.5)
    results = pd.concat(
        [
            results,
            pd.DataFrame(
                [[scib_bootstrapped, "combined_scib", tool_label, paired_rate, hyper]],
                columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"],
            ),
        ],
        ignore_index=True,
    )
    return results


In [ ]:
(
    mdata,
    mdata_counts,
    adata,
    train_indices,
    val_indices,
    n_genes,
    n_regions,
    n_snps,
    labels_key,
    further_labels_keys,
    batch_key,
    patient_key,
    further_batch_keys,
    further_continuous_batch_keys,
    n_patient_covariates,
    protein_expression_obsm_key,
) = datasets.load_neurips2021_cite_BMMC(1.0, None, True)


In [ ]:
def map_cell_type_to_family(cell_type):
    cell_type = cell_type.lower()

    if any(x in cell_type for x in ["hsc", "prog"]):
        return "Progenitor"
    elif "b" in cell_type:
        return "B cell"
    elif any(x in cell_type for x in ["nk", "ilc"]):
        return "ILC/NK"
    elif any(x in cell_type for x in ["cd4", "cd8", "treg", "dnT", "gdT", "mait"]):
        return "T cell"
    elif "mono" in cell_type or "mac" in cell_type:
        return "Monocyte"
    elif "dc" in cell_type:
        return "Dendritic cell"
    elif any(x in cell_type for x in ["erythro", "nomo", "retic", "normo"]):
        return "Erythroid"
    elif any(x in cell_type for x in ["plasmablast", "plasma"]):
        return "B cell"
    elif "platelet" in cell_type:
        return "Platelet"
    elif "doublet" in cell_type:
        return "Artifact"
    else:
        return "Other"

family_colors = {
    "Progenitor": "#4CAF50",
    "B cell": "#E53935",
    "ILC/NK": "#FB8C00",
    "T cell": "#FDD835",
    "Monocyte": "#F06292",
    "Dendritic cell": "#8E24AA",
    "Erythroid": "#6D4C41",
    "Platelet": "#90A4AE",
    "Artifact": "#BDBDBD",
    "Other": "#9E9E9E",
}

from matplotlib.colors import to_rgb, to_hex
import colorsys

def generate_family_shades(family_labels, base_color):
    '''Generate different shades for subtypes in a given family.'''
    base_rgb = to_rgb(base_color)
    hsv = colorsys.rgb_to_hsv(*base_rgb)
    n = len(family_labels)
    shades = []

    for i in range(n):
        value = 0.6 + (0.4 * (i / max(n - 1, 1)))
        rgb = colorsys.hsv_to_rgb(hsv[0], hsv[1], value)
        shades.append(to_hex(rgb))

    return dict(zip(family_labels, shades))

def generate_family_shades(family_labels, base_color, hue_shift=0.08):
    '''Generate visually distinct colors within a family by shifting hue slightly.'''
    base_rgb = to_rgb(base_color)
    h, s, v = colorsys.rgb_to_hsv(*base_rgb)
    n = len(family_labels)
    shades = []

    for i in range(n):
        new_h = (h + hue_shift * (i / max(n - 1, 1)) - hue_shift / 2) % 1.0
        new_s = min(max(s * (0.8 + 0.2 * (i / max(n - 1, 1))), 0), 1)
        new_v = min(max(v * (0.8 + 0.2 * (i / max(n - 1, 1))), 0), 1)
        rgb = colorsys.hsv_to_rgb(new_h, new_s, new_v)
        shades.append(to_hex(rgb))

    return dict(zip(family_labels, shades))

from collections import defaultdict

def build_color_map(cell_types):
    family_to_types = defaultdict(list)
    for ct in cell_types:
        family = map_cell_type_to_family(ct)
        family_to_types[family].append(ct)

    type_to_color = {}
    for family, types in family_to_types.items():
        base_color = family_colors.get(family, "#9E9E9E")
        shades = generate_family_shades(types, base_color)
        type_to_color.update(shades)

    return type_to_color

all_cell_types = list(np.unique(mdata.mod["rna"].obs[labels_key]))
type_to_color = build_color_map(all_cell_types)


# Ablation - Revision 2

In [ ]:
dataset_name = "neurips2021_cite_bmmc"
paired_rates = [1.0]

results = pd.DataFrame(columns=["scib", "scib_type", "tool_label", "paired_rate", "hyper"])

tools_labels = [
    "NetworkVI",
    "Pathway Commons",
    "Reactome",
    "GRAND GRN (K562)",
    "BioGRID",
    "STRING",
    "GM12878 (ENCFF203AKP)",
    "T47D (ENCFF804SET)",
    "MCF-7 (ENCFF164AGX)",
    "K562 Position-shifted TAD benchmark",
    "K562 interactions shuffle",
    "TFLink",
    "hTFtarget",
    "GO BP 2014",
    "GO BP 2019",
    "GO BP 2022",
    "Gene-GO ass. shuffle",
    "GO CC",
    "GO MF",
]

tools_paths = [
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/",
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_pcommons_filtont_pathway_s0/",
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_reactome_filtont_reactome_s0/",
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_grand_k562_grn_gaf_standard_with_rna_s0/",
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_biogrid_gaf_standard_with_rna_s0/",
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_string_gaf_standard_with_rna_s0/",
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF203AKP_gaf_standard_with_rna_s0/",
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF804SET_gaf_standard_with_rna_s0/",
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF164AGX_gaf_standard_with_rna_s0/",
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_shifted_benchmark_gaf_standard_with_rna_s0/",
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_shuff_gaf_standard_with_rna_s0/",
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_tflink_gaf_standard_with_rna_s0/",
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_htftarget_bm_gaf_standard_with_rna_s0/",
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_go2014_s0/",
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_go2019_s0/",
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_go2022_s0/",
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_gshuff_s0/",
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_filtont_cellular_component_s0/",
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_filtont_molecular_function_s0/",
]

tools_paths = [[path] for path in tools_paths]
tools_hyper = [True] * len(tools_labels)

print(len(tools_labels))
print(len(tools_hyper))

for tool_label, tool_hyper, tool_paths in tqdm(zip(tools_labels, tools_hyper, tools_paths)):
    for paired_rate, path in zip(paired_rates, tool_paths):
        if path is not None:

            results = get_scib_results(
                os.path.join(path, "default"), results, batch_key, tool_label, hyper=False
            )
            if tool_hyper:
                print(path)
                try:
                    num_hyper, path_scib_hyper, _ = get_best_hyper(path)
                    print(num_hyper)
                    if path_scib_hyper is not None:
                        results = get_scib_results(
                            path_scib_hyper, results, batch_key, tool_label, hyper=True
                        )
                except:
                    results = get_scib_results(
                        os.path.join(path, "default"), results, batch_key, tool_label, hyper=True
                    )


In [ ]:
results = results[(results["hyper"] == True) & (results["scib_type"] == "combined")]
results


In [ ]:
BASELINE_LABEL = 'NetworkVI'
networkvi_scib = results.loc[results['tool_label'] == BASELINE_LABEL, 'scib'].values[0]
SECTIONS = [('GO versions', ['GO BP 2014', 'GO BP 2019', 'GO BP 2022'], 'go'), ('GO\nname-\nspaces', ['GO MF', 'GO CC'], 'go'), ('Pathway\nontologies', ['Pathway Commons', 'Reactome'], 'pathway'), ('Non-specific\nnetworks', ['STRING', 'BioGRID', 'GRAND GRN (K562)', 'TFLink', 'hTFtarget'], 'tad'), ('Alt. cell-line\nTADs', ['GM12878 (ENCFF203AKP)', 'T47D (ENCFF804SET)', 'MCF-7 (ENCFF164AGX)'], 'tad'), ('Negative\nControl', ['K562 Position-shifted TAD benchmark', 'K562 interactions shuffle', 'Gene-GO ass. shuffle'], 'shuffle')]
TICK_RENAME = {'GM12878 (ENCFF203AKP)': 'GM12878 (lymphoid)', 'T47D (ENCFF804SET)': 'T47D (breast)', 'MCF-7 (ENCFF164AGX)': 'MCF-7 (breast)', 'K562 Position-shifted TAD benchmark': 'K562 Position-shifted'}
PALETTE = {'go': '#A0522D', 'pathway': '#5F8575', 'tad': '#6A5ACD', 'shuffle': '#D2691E'}
scib_lookup = results.set_index('tool_label')['scib'].to_dict()
bar_labels = []
bar_deltas = []
bar_colours = []
section_spans = {}
x = 0
for s_idx, (sec_name, tools, group) in enumerate(SECTIONS):
    x_start = x
    for tool in tools:
        scib_val = scib_lookup[tool]
        bar_labels.append(tool)
        bar_deltas.append(scib_val - networkvi_scib)
        bar_colours.append(PALETTE[group])
        x += 1
    x_end = x - 1
    section_spans[sec_name] = (x_start, x_end)
n_bars = len(bar_labels)
xs = np.arange(n_bars)
fig_width = 1 / 2 * 7.0
fig_height = 2
set_figure_params(fig_width, fig_height)
fig, ax = plt.subplots()
Y_CLIP_MIN = -0.2
bar_deltas_clipped = [max(d, Y_CLIP_MIN) for d in bar_deltas]
needs_break = [d < Y_CLIP_MIN for d in bar_deltas]
bar_width = 0.6
bars = ax.bar(xs, bar_deltas_clipped, color=bar_colours, width=bar_width, edgecolor='white', linewidth=0.3, zorder=4)
for xi, (clipped, broken, delta) in enumerate(zip(bar_deltas_clipped, needs_break, bar_deltas)):
    if broken:
        slash_y = Y_CLIP_MIN + 0.003
        for offset in (-0.08, 0.08):
            ax.plot([xi + offset - 0.06, xi + offset + 0.06], [slash_y - 0.008, slash_y + 0.008], color='white', linewidth=1.2, zorder=6)
            ax.plot([xi + offset - 0.06, xi + offset + 0.06], [slash_y - 0.016, slash_y], color=bar_colours[xi], linewidth=1.2, zorder=6)
ax.axhline(0, color='black', linewidth=0.8, zorder=3)
trans = ax.get_xaxis_transform()
BRACKET_Y = 1.06
TICK_DROP = 0.03
LABEL_Y = 1.1
SECTION_DISPLAY = {'GO versions': 'GO versions', 'GO\nnamespaces': 'GO\nnamespaces', 'Pathway\nontologies': 'Pathway\nontologies', 'Non-specific\nnetworks': 'Non-specific\nnetworks', 'Alt. cell-line\nTADs': 'Alt. cell-line\nTADs', 'Shuffle\ncontrol': 'Control'}

def bracket_half(x_start, x_end):
    n = x_end - x_start + 1
    return 0.2 if n <= 2 else 0.28
for sec_name, (x_start, x_end) in section_spans.items():
    bh = bracket_half(x_start, x_end)
    lx = x_start - bh
    rx = x_end + bh
    mid = (lx + rx) / 2.0
    label = SECTION_DISPLAY.get(sec_name, sec_name)
    ax.plot([lx, rx], [BRACKET_Y, BRACKET_Y], transform=trans, color='#555555', linewidth=0.7, clip_on=False, zorder=10)
    ax.plot([lx, lx], [BRACKET_Y - TICK_DROP, BRACKET_Y], transform=trans, color='#555555', linewidth=0.7, clip_on=False, zorder=10)
    ax.plot([rx, rx], [BRACKET_Y - TICK_DROP, BRACKET_Y], transform=trans, color='#555555', linewidth=0.7, clip_on=False, zorder=10)
    ax.text(mid, LABEL_Y, label, transform=trans, ha='center', va='bottom', fontsize=mpl.rcParams['font.size'] - 0.5, color='#333333', multialignment='center', linespacing=1.2, clip_on=False)
tick_labels = [TICK_RENAME.get(l, l) for l in bar_labels]
ax.set_xticks(xs)
ax.set_xticklabels(tick_labels, rotation=40, ha='right')
ax.set_ylabel('ΔscIB score vs. NetworkVI')
delta_min_vis = min(bar_deltas_clipped)
delta_max_vis = 0.05
rng = delta_max_vis - delta_min_vis
ax.set_ylim(delta_min_vis - rng * 0.18, delta_max_vis)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(-0.75, n_bars - 0.25)
out_pdf = 'networkvi_ablation_prior_knowledge.pdf'
out_png = 'networkvi_ablation_prior_knowledge.png'
out_svg = 'networkvi_ablation_prior_knowledge.svg'
fig.savefig(os.path.join(_OUTDIR_ABS, out_pdf), dpi=500, bbox_inches='tight', transparent=True)
fig.savefig(os.path.join(_OUTDIR_ABS, out_png), dpi=500, bbox_inches='tight', transparent=True)
fig.savefig(os.path.join(_OUTDIR_ABS, out_svg), format='svg', bbox_inches='tight', transparent=True)
print(f'Saved:\n  {out_pdf}\n  {out_png}\n  {out_svg}')


In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

def analyze_graph_pkls(base_dir, threshold=0.55, verbose=True):
    '''Find all *_graph_csv_with_ci.pkl files under base_dir,'''

    results = []

    for root, _, files in os.walk(base_dir):
        for f in tqdm(files):
            if f.endswith("_graph_csv_with_ci.pkl"):
                path = os.path.join(root, f)

                try:
                    df = pd.read_pickle(path)

                    total_terms = len(df)
                    if total_terms == 0:
                        continue

                    frac_above = (df["predictors_label_valid_roc"] > threshold).sum() / total_terms

                    results.append(
                        {"file": path, "n_terms": total_terms, "frac_above_threshold": frac_above}
                    )

                    if verbose:
                        print(f"\n{path}")
                        print(f"  Terms: {total_terms}")
                        print(f"  Fraction > {threshold}: {frac_above:.4f}")

                except Exception as e:
                    print(f"Failed to load {path}: {e}")

    if not results:
        print("No matching files found.")
        return [], None, None

    mean_fraction = np.mean([r["frac_above_threshold"] for r in results])
    mean_terms = np.mean([r["n_terms"] for r in results])

    print("\n=== SUMMARY ===")
    print(f"Files analyzed: {len(results)}")
    print(f"Mean #terms: {mean_terms:.2f}")
    print(f"Mean fraction > {threshold}: {mean_fraction:.4f}")

    return results, mean_fraction, mean_terms


In [ ]:
base_path = "../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_pcommons_filtont_pathway_s0/default/calc_go_terms_cell_type/"
results, mean_frac, mean_terms = analyze_graph_pkls(base_path)


In [ ]:
base_path = "../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_reactome_filtont_reactome_s0/"
results, mean_frac, mean_terms = analyze_graph_pkls(base_path)


In [ ]:
base_path = "../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_filtont_molecular_function_s0/"
results, mean_frac, mean_terms = analyze_graph_pkls(base_path)


In [ ]:
base_path = "../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_filtont_cellular_component_s0/"
results, mean_frac, mean_terms = analyze_graph_pkls(base_path)


# Stability

In [ ]:
graph_csv = pd.read_pickle(
    "../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_subsam_0.5_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_go_terms_cell_type/expression_cell_type_vs_rest_B1 B IGKC+_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv_with_ci.pkl"
)
graph_csv


In [ ]:
graph_csv1 = pd.read_pickle(
    "../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_go_terms_cell_type/expression_cell_type_vs_rest_B1 B IGKC+_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv_with_ci.pkl"
)
graph_csv1


In [ ]:
go_one_over055 = list(
    set(graph_csv[graph_csv["predictors_label_valid_roc"] > 0.65].index)
    & set(graph_csv1[graph_csv1["predictors_label_valid_roc"] > 0.65].index)
)


In [ ]:
scipy.stats.spearmanr(
    graph_csv["predictors_label_valid_roc"], graph_csv1["predictors_label_valid_roc"]
)


In [ ]:
scipy.stats.spearmanr(
    graph_csv["predictors_label_valid_roc"].loc[go_one_over055],
    graph_csv1["predictors_label_valid_roc"].loc[go_one_over055],
)


In [ ]:
"""Full stability analysis: all pairwise comparisons across seeds and subsampling."""
import os, re, glob
import numpy as np
import pandas as pd
from itertools import combinations
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import spearmanr
_BASE = '../.experiment_data/scratch/experiments_new/'
_SEED = 'sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_{s}/default/calc_go_terms_cell_type/'
_SUB = 'sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_subsam_{f}_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_go_terms_cell_type/'
MODELS = [('Seed 0\n(base)', _BASE + _SEED.format(s='s0'), 'seed'), ('Seed 1', _BASE + _SEED.format(s='s1'), 'seed'), ('Seed 2', _BASE + _SEED.format(s='s2'), 'seed'), ('50 % cells', _BASE + _SUB.format(f='0.5'), 'subsample'), ('75 % cells', _BASE + _SUB.format(f='0.75'), 'subsample')]
CELL_TYPES_FILTER = ['CD8+ T CD49f+', 'CD8+ T TIGIT+ CD45RO+', 'Erythroblast', 'Normoblast', 'CD14+ Mono', 'CD16+ Mono', 'Naive CD20+ B IGKC+', 'NK', 'pDC', 'CD4+ T naive', 'CD8+ T naive']
GROUP_PALETTE = {'seed': '#8B6BAE', 'subsample': '#5F8575'}
N_BOOT, RANDOM_STATE = (5000, 42)
AUROC_COL = 'predictors_label_valid_roc'

def _canonical(name: str) -> str:
    """'G/M prog' == 'G_M prog' == 'G_M_prog'  →  'g m prog'"""
    s = name.lower()
    s = re.sub('[/_]', ' ', s)
    s = re.sub(' +', ' ', s).strip()
    return s
_PKL_RE = re.compile('^(expression|protein)_cell_type_(vs_rest|vs_other_t_cells)_(.+?)_res_unp_act_mask_False_flip_False_go_terms(?:_genes)?_evaluated_graph_csv_with_ci\\.pkl$')

def _parse(fname):
    m = _PKL_RE.match(os.path.basename(fname))
    return (m.group(1), m.group(2), m.group(3)) if m else None

def load_model(calc_dir, filter_phenotypes=None, modality='expression', comparison_type='vs_rest', verbose=True):
    """Load all matching pkls → {canonical_key: DataFrame}."""
    print(calc_dir)
    pkls = sorted(glob.glob(os.path.join(calc_dir, '*.pkl')))
    if not pkls:
        raise FileNotFoundError(f'No pkls in: {calc_dir}')
    filter_canon = {_canonical(p) for p in filter_phenotypes} if filter_phenotypes else None
    out = {}
    for path in pkls:
        parsed = _parse(path)
        if not parsed:
            continue
        mod, ctype, raw = parsed
        if modality is not None and mod != modality:
            continue
        if comparison_type is not None and ctype != comparison_type:
            continue
        canon = _canonical(raw)
        if filter_canon is not None and canon not in filter_canon:
            continue
        try:
            df = pd.read_pickle(path)
        except Exception as e:
            if verbose:
                print(f'  ⚠ {os.path.basename(path)}: {e}')
            continue
        for col in ['go_term', 'GO_term', 'go_id', 'term']:
            if col in df.columns:
                df = df.set_index(col)
                break
        out[canon] = df
    if verbose:
        label = os.path.basename(calc_dir.rstrip('/'))[-30:]
        print(f'  {len(out):2d} phenotypes  ←  ...{label}')
    return out

def spearman_bootstrap(x, y, n_boot=N_BOOT, rs=RANDOM_STATE):
    rng = np.random.default_rng(rs)
    x, y = (np.asarray(x, float), np.asarray(y, float))
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = (x[ok], y[ok])
    n = len(x)
    if n < 5:
        return dict(rho=np.nan, ci_lo=np.nan, ci_hi=np.nan, p_value=np.nan, n=n)
    rho_obs, _ = spearmanr(x, y)
    corrs = [spearmanr(x[rng.choice(n, n, replace=True)], y[rng.choice(n, n, replace=True)])[0] for _ in range(n_boot)]
    corrs = np.array([c for c in corrs if np.isfinite(c)])
    cent = corrs - corrs.mean()
    p = max(float(2 * min(np.mean(cent >= abs(rho_obs)), np.mean(cent <= -abs(rho_obs)))), 1 / n_boot)
    return dict(rho=float(rho_obs), ci_lo=float(np.percentile(corrs, 2.5)), ci_hi=float(np.percentile(corrs, 97.5)), p_value=p, n=n)

def compare_pair(data_a, data_b, min_go=10):
    """Compute per-phenotype Spearman ρ for one model pair."""
    shared = sorted(set(data_a) & set(data_b))
    rows = []
    for ph in shared:
        shared_go = data_a[ph].index.intersection(data_b[ph].index)
        if len(shared_go) < min_go:
            continue
        if AUROC_COL not in data_a[ph].columns:
            continue
        x = data_a[ph].loc[shared_go, AUROC_COL].values.astype(float)
        y = data_b[ph].loc[shared_go, AUROC_COL].values.astype(float)
        s = spearman_bootstrap(x, y)
        s['phenotype'] = ph
        s['n_go'] = len(shared_go)
        rows.append(s)
    return pd.DataFrame(rows).set_index('phenotype') if rows else pd.DataFrame()

def _stars(p):
    if np.isnan(p):
        return ''
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return 'ns'

def load_all_models(models, filter_phenotypes, verbose=True):
    """Returns list of (label, group, data_dict)."""
    loaded = []
    for label, path, group in models:
        if verbose:
            print(f'Loading {label!r} …')
        try:
            data = load_model(path, filter_phenotypes=filter_phenotypes, verbose=verbose)
            loaded.append((label, group, data))
        except FileNotFoundError as e:
            print(f'  ⚠ skipped: {e}')
    return loaded

def plot_vs_base(loaded_models, base_label='Seed 0\n(base)', group_palette=GROUP_PALETTE, display_labels=None, fig_width=3 / 4 * 7.0, fig_height=2.6):
    '''Bar chart: Spearman ρ vs the base model, one bar per non-base model,'''
    base_data = next((d for l, g, d in loaded_models if l == base_label), None)
    if base_data is None:
        raise ValueError(f"Base model '{base_label}' not found in loaded_models.")
    comparisons = [(l, g, d) for l, g, d in loaded_models if l != base_label]
    if not comparisons:
        raise ValueError('No comparison models found.')
    all_ph_canon = sorted(set().union(*[set(d) for _, _, d in loaded_models]))
    n_comp = len(comparisons)
    n_pheno = len(all_ph_canon)
    rho_mat = np.full((n_comp, n_pheno), np.nan)
    lo_mat = np.full_like(rho_mat, np.nan)
    hi_mat = np.full_like(rho_mat, np.nan)
    p_mat = np.full_like(rho_mat, np.nan)
    for i, (label, group, data) in enumerate(comparisons):
        df = compare_pair(base_data, data)
        for j, ph in enumerate(all_ph_canon):
            if ph in df.index:
                rho_mat[i, j] = df.loc[ph, 'rho']
                lo_mat[i, j] = df.loc[ph, 'ci_lo']
                hi_mat[i, j] = df.loc[ph, 'ci_hi']
                p_mat[i, j] = df.loc[ph, 'p_value']
    order = np.argsort(-np.nanmean(rho_mat, axis=0))
    all_ph_canon = [all_ph_canon[k] for k in order]
    rho_mat = rho_mat[:, order]
    lo_mat = lo_mat[:, order]
    hi_mat = hi_mat[:, order]
    p_mat = p_mat[:, order]
    if display_labels:
        tick_labels = [display_labels.get(ph, ph) for ph in all_ph_canon]
    else:
        tick_labels = all_ph_canon
    mpl.rcParams['figure.figsize'] = (fig_width, fig_height)
    fig, ax = plt.subplots()
    bar_w = 0.72 / n_comp
    offsets = np.linspace(-(n_comp - 1) / 2, (n_comp - 1) / 2, n_comp) * bar_w
    xs = np.arange(n_pheno)
    for i, (label, group, _) in enumerate(comparisons):
        colour = group_palette.get(group, '#888888')
        xp = xs + offsets[i]
        rho_v, lo_v, hi_v, p_v = (rho_mat[i], lo_mat[i], hi_mat[i], p_mat[i])
        err_lo = np.clip(np.where(np.isnan(lo_v), 0, rho_v - lo_v), 0, None)
        err_hi = np.clip(np.where(np.isnan(hi_v), 0, hi_v - rho_v), 0, None)
        ax.bar(xp, rho_v, width=bar_w * 0.88, color=colour, alpha=0.88, edgecolor='white', linewidth=0.25, zorder=4, label=label)
        ax.errorbar(xp, rho_v, yerr=[err_lo, err_hi], fmt='none', ecolor='#333333', elinewidth=0.7, capsize=1.8, capthick=0.7, zorder=5)
        for j in range(n_pheno):
            if np.isnan(rho_v[j]):
                continue
            top = hi_v[j] if np.isfinite(hi_v[j]) else rho_v[j]
            star_y = top + 0.012 + i * 0.055
            ax.text(xp[j], star_y, _stars(p_v[j]), ha='center', va='bottom', fontsize=mpl.rcParams['font.size'] - 1, color='#333333', zorder=6)
    ax.axhline(1.0, color='#aaaaaa', lw=0.5, ls='--', zorder=1)
    ax.axhline(0.0, color='black', lw=0.6, zorder=2)
    ax.set_xticks(xs)
    ax.set_xticklabels(tick_labels, rotation=40, ha='right')
    ax.set_ylabel('Spearman ρ vs. Seed 0 (base)')
    ax.set_xlim(-0.6, n_pheno - 0.4)
    y_top = float(np.nanmax(hi_mat[np.isfinite(hi_mat)], initial=1)) + 0.015 + n_comp * 0.055 + 0.1
    ax.set_ylim(-0.05, min(y_top, 1.45))
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    seen = {}
    for label, group, _ in comparisons:
        seen.setdefault(group, []).append(label)
    handles = []
    for group, labels in seen.items():
        colour = group_palette.get(group, '#888')
        handles.append(mpatches.Patch(facecolor=colour, alpha=0.88, label=', '.join((l.replace('\n', ' ') for l in labels))))
    ax.legend(handles=handles, loc='lower left', frameon=False, handlelength=0.9, handletextpad=0.4)
    y_guide = min(y_top - 0.02, 1.4)
    for k, txt in enumerate(['*** p<0.001', '** p<0.01', '* p<0.05', 'ns p≥0.05']):
        ax.text(n_pheno - 0.08, y_guide - k * 0.075, txt, ha='right', va='top', fontsize=mpl.rcParams['font.size'] - 1.5, color='#555555')
    fig.tight_layout()
    return fig

def _pairwise_matrix(loaded_models):
    """Compute (n_models × n_models) matrix of mean Spearman ρ across phenotypes."""
    n = len(loaded_models)
    labels = [l for l, g, d in loaded_models]
    rho_mat = np.full((n, n), np.nan)
    np.fill_diagonal(rho_mat, 1.0)
    for (i, (la, ga, da)), (j, (lb, gb, db)) in combinations(enumerate(loaded_models), 2):
        df = compare_pair(da, db)
        if not df.empty and 'rho' in df.columns:
            mean_rho = float(df['rho'].dropna().mean())
            rho_mat[i, j] = mean_rho
            rho_mat[j, i] = mean_rho
    return (labels, rho_mat)

def plot_pairwise_heatmap(loaded_models, group_palette=GROUP_PALETTE, fig_width=3.2, fig_height=3.0):
    '''Heatmap of mean Spearman ρ across all cell types for every model pair.'''
    labels, rho_mat = _pairwise_matrix(loaded_models)
    groups = [g for l, g, d in loaded_models]
    n = len(labels)
    mpl.rcParams['figure.figsize'] = (fig_width, fig_height)
    fig, ax = plt.subplots()
    from matplotlib.colors import LinearSegmentedColormap
    cmap = LinearSegmentedColormap.from_list('rho_cmap', ['#f7f7f7', '#3d6b6e'], N=256)
    vmin, vmax = (0.75, 1.0)
    im = ax.imshow(rho_mat, cmap=cmap, vmin=vmin, vmax=vmax, aspect='equal')
    fs = mpl.rcParams['font.size'] - 0.5
    for i in range(n):
        for j in range(n):
            val = rho_mat[i, j]
            if np.isfinite(val):
                txt_col = 'white' if val > vmin + (vmax - vmin) * 0.6 else '#333333'
                ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=fs, color=txt_col)
    clean = [l.replace('\n', ' ') for l in labels]
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(clean, rotation=40, ha='right', fontsize=fs)
    ax.set_yticklabels(clean, fontsize=fs)
    for tick, grp in zip(ax.get_xticklabels(), groups):
        tick.set_color(group_palette.get(grp, '#333333'))
    for tick, grp in zip(ax.get_yticklabels(), groups):
        tick.set_color(group_palette.get(grp, '#333333'))
    cb = fig.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
    cb.set_label('mean Spearman ρ', fontsize=mpl.rcParams['font.size'] - 0.5)
    cb.ax.tick_params(labelsize=mpl.rcParams['font.size'] - 1)
    ax.spines[:].set_visible(False)
    ax.tick_params(length=0)
    fig.tight_layout()
    return fig

def pairwise_summary(loaded_models, display_labels=None):
    '''Tidy DataFrame: one row per (model_A, model_B, phenotype).'''
    rows = []
    for (la, ga, da), (lb, gb, db) in combinations(loaded_models, 2):
        df = compare_pair(da, db)
        if df.empty:
            continue
        df = df.copy()
        df['model_a'] = la.replace('\n', ' ')
        df['model_b'] = lb.replace('\n', ' ')
        df['stars'] = df['p_value'].apply(_stars)
        if display_labels:
            df.index = [display_labels.get(p, p) for p in df.index]
        rows.append(df.reset_index().rename(columns={'index': 'phenotype'}))
    if not rows:
        return pd.DataFrame()
    out = pd.concat(rows, ignore_index=True)
    cols = ['model_a', 'model_b', 'phenotype', 'rho', 'ci_lo', 'ci_hi', 'p_value', 'stars', 'n', 'n_go']
    return out[[c for c in cols if c in out.columns]]
if __name__ == '__main__':
    os.makedirs(_OUTDIR_ABS, exist_ok=True)
    print('Loading models …')
    loaded = load_all_models(MODELS, filter_phenotypes=CELL_TYPES_FILTER)
    display_labels = {_canonical(p): p for p in CELL_TYPES_FILTER}
    tbl = pairwise_summary(loaded, display_labels=display_labels)
    print(tbl.to_string(index=False))
    print('Done.')
